In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

print("✓ Google Drive mounted!")
print("Your drive is accessible at: /content/drive/MyDrive/")

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
MODEL_PATH = 'Qwen/Qwen2.5-0.5B-Instruct'
output_dir = "./Colab Notebooks/qwen-cybersec-pretrained/modelntoken2"
os.makedirs(output_dir, exist_ok=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('./Colab Notebooks/qwen-cybersec-pretrained/modelntoken2\\tokenizer_config.json',
 './Colab Notebooks/qwen-cybersec-pretrained/modelntoken2\\special_tokens_map.json',
 './Colab Notebooks/qwen-cybersec-pretrained/modelntoken2\\chat_template.jinja',
 './Colab Notebooks/qwen-cybersec-pretrained/modelntoken2\\vocab.json',
 './Colab Notebooks/qwen-cybersec-pretrained/modelntoken2\\merges.txt',
 './Colab Notebooks/qwen-cybersec-pretrained/modelntoken2\\added_tokens.json',
 './Colab Notebooks/qwen-cybersec-pretrained/modelntoken2\\tokenizer.json')

In [ ]:
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer
import torch
import os

# ========================================
# 🔧 CONFIGURATION
# ========================================
MODEL_PATH = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_LENGTH = 512  # adjust based on GPU memory
BATCH_SIZE = 4    # reduce if OOM on RTX 4050 (6GB VRAM)
LEARNING_RATE = 1e-5 # Reduced learning rate for stability
NUM_EPOCHS = 3
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/qwen-cybersec-pretrained'
TOKENIZED_DATA_DIR = './Colab Notebooks/tokenized_datasets'  # New folder for tokenized data

print(f"✓ Training config loaded")
print(f"  Model: {MODEL_PATH}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Batch size: {BATCH_SIZE}")

# Create directory for tokenized datasets
os.makedirs(TOKENIZED_DATA_DIR, exist_ok=True)

# ========================================
# 📦 LOAD ALL SUBSETS
# ========================================
print("\n📦 Loading Primus-Seed datasets...")

subsets = [
    "default",
    # "cybersecurity_companies_websites",
    # "cybersecurity_wikis",
    # "mitre"
]

# ========================================
# 🔤 TOKENIZER & PREPROCESSING
# ========================================
print("\n🔤 Loading tokenizer...")
    

# Set pad token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

all_datasets = []
all_tokenized_datasets = []

for subset in subsets:
    print(f"\n{'='*60}")
    print(f"📂 Processing subset: {subset}")
    print(f"{'='*60}")
    
    # Load dataset
    print(f"  Loading {subset}...")
    ds = load_dataset("trendmicro-ailab/Primus-Seed", subset)

    # Take train split (or concatenate train+test if you want all data)
    if 'train' in ds:
        dataset = ds['train']
    else:
        # If no train split, use the first available split
        split_name = list(ds.keys())[0]
        dataset = ds[split_name]

    print(f"    ✓ Loaded {len(dataset)} examples")
    print(f"    ✓ Columns: {dataset.column_names}")
    
    all_datasets.append(dataset)
    
    # Define preprocessing function for this subset
    def preprocess_function(examples):
        """
        Concatenate all text fields and tokenize for causal LM.
        """
        # Identify text columns (skip IDs, URLs, etc.)
        text_columns = []
        for col in dataset.column_names:
            if col in ['description', 'text', 'content', 'name', 'technique']:
                text_columns.append(col)

        # Concatenate all text fields with separators
        texts = []
        for i in range(len(examples[text_columns[0]])):
            parts = []
            for col in text_columns:
                if examples[col][i]:  # skip None/empty
                    parts.append(str(examples[col][i]))
            # Join with newlines
            texts.append("\n\n".join(parts))

        # Tokenize
        tokenized = tokenizer(
            texts,
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt"
        )
        
        # For causal LM, labels = input_ids (shifted internally by model)
        tokenized["labels"] = tokenized["input_ids"].clone()

        return tokenized
    
    # Tokenize this subset
    print(f"  🔄 Tokenizing {subset}...")
    tokenized_dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names,
        desc=f"Tokenizing {subset}"
    )
    
    print(f"    ✓ Tokenized {len(tokenized_dataset):,} examples")
    
    # Save tokenized dataset
    subset_save_path = os.path.join(TOKENIZED_DATA_DIR, subset)
    print(f"  💾 Saving tokenized dataset to: {subset_save_path}")
    tokenized_dataset.save_to_disk(subset_save_path)
    print(f"    ✓ Saved!")
    
    all_tokenized_datasets.append(tokenized_dataset)

# Combine all tokenized datasets
print(f"\n{'='*60}")
print("🔗 Combining all tokenized datasets...")
print(f"{'='*60}")
combined_dataset = concatenate_datasets(all_tokenized_datasets)
print(f"✓ Total tokenized examples: {len(combined_dataset):,}")

# Save combined dataset
combined_save_path = os.path.join(TOKENIZED_DATA_DIR, "combined_all_subsets")
print(f"\n💾 Saving combined dataset to: {combined_save_path}")
combined_dataset.save_to_disk(combined_save_path)
print(f"✓ Combined dataset saved!")

# Assign to tokenized_dataset for training
tokenized_dataset = combined_dataset

print(f"\n{'='*60}")
print("✅ ALL DATASETS TOKENIZED AND SAVED")
print(f"{'='*60}")
print(f"📁 Tokenized data location: {TOKENIZED_DATA_DIR}")
print(f"   Individual subsets: {subsets}")
print(f"   Combined dataset: combined_all_subsets")
print(f"✓ Dataset ready for training!")

✓ Training config loaded
  Model: Qwen/Qwen2.5-1.5B-Instruct
  Max length: 512
  Batch size: 4

📦 Loading Primus-Seed datasets...

🔤 Loading tokenizer...

📂 Processing subset: default
  Loading default...
    ✓ Loaded 86987 examples
    ✓ Columns: ['time', 'source', 'url', 'content']
  🔄 Tokenizing default...
    ✓ Tokenized 86,987 examples
  💾 Saving tokenized dataset to: ./Colab Notebooks/tokenized_datasets\default


Saving the dataset (2/2 shards): 100%|██████████| 86987/86987 [00:01<00:00, 58512.30 examples/s] 


    ✓ Saved!

🔗 Combining all tokenized datasets...
✓ Total tokenized examples: 86,987

💾 Saving combined dataset to: ./Colab Notebooks/tokenized_datasets\combined_all_subsets


Saving the dataset (2/2 shards): 100%|██████████| 86987/86987 [00:02<00:00, 41091.02 examples/s]

✓ Combined dataset saved!

✅ ALL DATASETS TOKENIZED AND SAVED
📁 Tokenized data location: ./Colab Notebooks/tokenized_datasets
   Individual subsets: ['default']
   Combined dataset: combined_all_subsets
✓ Dataset ready for training!


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
import torch
from torch.utils.data import DataLoader

print("\n🧠 Loading model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device: {device}")

if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Clear cache before loading
    torch.cuda.empty_cache()

# Load model with optimizations
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
    device_map="auto",  # Automatically handle device placement
)

# IMPORTANT: Enable gradient checkpointing ONLY on model, not in TrainingArguments
# This fixes the FP16 unscaling error
# if device == "cuda":
#     model.gradient_checkpointing_enable()
#     # Some models need this additional config
#     if hasattr(model, 'enable_input_require_grads'):
#         model.enable_input_require_grads()

print("✓ Model loaded")


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM (not masked LM)
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    
    # Training hyperparameters - adjusted for T4
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,  # Reduced from 4 to 1
    gradient_accumulation_steps=16,  # Increased to maintain effective batch size
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=500,
    max_grad_norm=1.0,  # Gradient clipping for stability
    
    # Memory optimization - crucial for T4
    fp16=True if device == "cuda" else False,
    fp16_full_eval=False,  # Disable FP16 for evaluation to avoid issues
    
    # CRITICAL: Set to False since we enabled it on model directly
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False} if device == "cuda" else None,
    
    # Optimizer - fallback to standard if fused not available
    optim="adamw_torch",  # Changed from adamw_torch_fused for stability
    
    # DataLoader optimization
    dataloader_num_workers=2,  # Parallel data loading
    dataloader_pin_memory=True,  # Faster data transfer to GPU
    
    # Logging & checkpointing
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,  # Keep only 2 checkpoints to save disk space
    
    # Memory management
    auto_find_batch_size=False,  # Set True to auto-find optimal batch size
    
    # Evaluation (optional)
    # evaluation_strategy="steps",
    # eval_steps=500,
    # per_device_eval_batch_size=1,
    
    # Other
    remove_unused_columns=False,
    report_to="none",  # Use "tensorboard" for monitoring in Colab
    disable_tqdm=False,  # Show progress bars
    load_best_model_at_end=False,  # Saves memory
)

print("✓ Training arguments configured")


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("\n" + "="*60)
print("🚀 TRAINING READY (T4 OPTIMIZED - FP16 FIX)")
print("="*60)
print(f"Total examples: {len(tokenized_dataset):,}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size per device: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Steps per epoch: ~{len(tokenized_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print(f"Total steps: ~{len(tokenized_dataset) * NUM_EPOCHS // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print(f"Optimizer: {training_args.optim}")
print(f"FP16: {training_args.fp16}")
print(f"Gradient Checkpointing: Enabled on model (not TrainingArguments)")
print("="*60 + "\n")

if device == "cuda":
    print("💡 Memory before training:")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    print()



🧠 Loading model...
  Device: cuda
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU
  VRAM: 6.4 GB


Some parameters are on the meta device because they were offloaded to the cpu and disk.
C:\Users\ADITYA RM\AppData\Local\Temp\ipykernel_114644\226040792.py:96: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


✓ Model loaded
✓ Training arguments configured


The model is already on multiple devices. Skipping the move to device specified in `args`.



🚀 TRAINING READY (T4 OPTIMIZED)
Total examples: 86,987
Epochs: 3
Batch size per device: 1
Gradient accumulation steps: 16
Effective batch size: 16
Steps per epoch: ~5436
Total steps: ~16310
Optimizer: OptimizerNames.ADAMW_TORCH_FUSED
FP16: True
Gradient Checkpointing: True

💡 Memory before training:
  Allocated: 6.20 GB
  Reserved: 6.49 GB



In [5]:
import time

print("🏋️ Starting training...")
start_time = time.time()

try:
    trainer.train()
    
    training_time = time.time() - start_time
    print(f"\n✅ Training completed in {training_time/60:.1f} minutes!")
    
    # Save final model
    print(f"\n💾 Saving model to {OUTPUT_DIR}...")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("✓ Model saved!")
    
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    raise

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🏋️ Starting training...


KeyboardInterrupt: 